In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import sys
sys.path.insert(0, "/Users/jihaein/Developer/investment-evidence-system")

In [3]:
from backtest.engine import load_daily_price
from backtest.montecarlo import run_montecarlo
import os
from dotenv import load_dotenv
load_dotenv("/Users/jihaein/Developer/investment-evidence-system/.env")

base = os.getenv("BACKTEST_DATA_DIR")
grid = load_daily_price()
print("grid:", grid.shape, " base set:", bool(base))


grid: (1249, 120)  base set: True


In [4]:
from backtest.montecarlo import _verdict, DEFAULT_BASELINE_DATE
from models.seoseohi.pipeline import load_ksic_from_base
from models.seoseohi.fscore import list_snapshot_dates

ksic = load_ksic_from_base(base)
dates = [d for d in list_snapshot_dates(base) if d > DEFAULT_BASELINE_DATE]
for s in grid.columns:
    for d in dates:
        _verdict(base, ksic, s, DEFAULT_BASELINE_DATE, d)
        
print("cache warmed:", len(__import__('backtest.montecarlo', fromlist=['_VERDICT_CACHE'])._VERDICT_CACHE))

cache warmed: 2280


## 핵심 — 네 전략 비교 (2년 · 4년, 전체 지표)
무손절 / 단순손절 / 서서히(개수) / 서서히(합산 4점).
지표: 누적수익률 · 수익률_중앙값 · MDD · Sortino · Sharpe · 평균_현금대체 · 평균_매도수 · 적중률 · 승률.


In [ ]:
# 네 전략 비교 (서서히 개수 vs 서서히 합산 4점) — PM 2026-11-13 규칙 반영
# with_sum=True 면 '서서히_합산'(세 점수 합계>=4 매도) 전략이 추가된다.
for yrs in [2, 4]:
    detail, summary = run_montecarlo(grid, base, years=yrs, seed=0, with_sum=True)
    print(f"=== {yrs}년 (seed=0, 바스켓 10개) ===")
    print(summary.round(3).to_string())
    print()

## 아웃오브샘플 검증 — 엣지가 일반화되는가
규칙은 120종목·2016~2020 전체로 맞춰졌으므로, 아래는 **엄밀한 train/test가 아니라 '쏠림/안정성' 검사**다.
엣지가 두 종목 그룹·두 기간 모두에서 유지되면 견고하고, 한쪽에만 몰리면 과적합 신호다.
(진짜 out-of-sample은 데이터팀의 새 종목/기간이 필요하다.)


In [ ]:
# 아웃오브샘플 1: 종목 반분 — 엣지가 두 그룹 모두에서 유지되나?
import numpy as np
codes = sorted(grid.columns)
perm = np.random.default_rng(42).permutation(codes)
halves = {"A조 (60종목)": sorted(perm[:60]), "B조 (60종목)": sorted(perm[60:])}
for name, half in halves.items():
    _, s = run_montecarlo(grid[half], base, years=4, seed=0, n_pick=20, n_runs=10, with_sum=True)
    print(f"===== {name} · 4년 =====")
    print(s.round(3).to_string())
    print()

In [ ]:
# 아웃오브샘플 2: 다른 기간 — 2018년 진입(후반부)에서도 엣지가 유지되나?
for label, entry, baseline in [("2016 진입 (원래)",  "2016-01-04", "2016-04-01"),
                               ("2018 진입 (후반부)", "2018-01-02", "2018-04-01")]:
    _, s = run_montecarlo(grid, base, years=2, seed=0,
                          entry_date=entry, baseline_date=baseline, with_sum=True)
    print(f"===== {label} · 2년 =====")
    print(s.round(3).to_string())
    print()


## 매도 품질 — 종목별 · 점수별, 그리고 정직한 집계
아래 표는 120종목 전체 매도와 D/F/O 점수. 그 다음 셀은 078130 같은 **희석 이상치**가
평균을 왜곡하므로 원시 · winsorize · 중앙값을 함께 보여준다.


In [ ]:
# 120종목 전체 서서히 매도 표 (개수 방식 / 합산 4점 비교, 세 모델 점수 포함) — PM 2026-11-13
import os, pandas as pd
from backtest.montecarlo import (seoseohi_schedule, _scores,
                                 DEFAULT_BASELINE_DATE, DEFAULT_ENTRY_DATE, DEFAULT_TRIGGER)
from models.seoseohi.pipeline import integrate, integrate_sum, load_ksic_from_base
from models.seoseohi.fscore import list_snapshot_dates
from backtest.engine import CASH_DAILY_RATE

YEARS = 4                      # None 이면 2020-12 까지
MODES = ["count", "sum"]
GOOD, BAD = "좋은매도", "성급한매도"

ksic = load_ksic_from_base(base)
entry = pd.Timestamp(DEFAULT_ENTRY_DATE)
end_date = None if YEARS is None else entry + pd.DateOffset(years=YEARS)

nm = pd.read_csv(os.path.join(base, "12_기업개황_KSIC.csv"),
                 header=0, skiprows=[1], dtype=str, encoding="utf-8-sig")
nm["stock_code"] = nm["stock_code"].str.zfill(6)
name_map = dict(zip(nm["stock_code"], nm["corp_name"]))

all_codes = list(grid.columns)
eval_dates = [d for d in list_snapshot_dates(base)
              if pd.Timestamp(d) > pd.Timestamp(DEFAULT_BASELINE_DATE)
              and (end_date is None or pd.Timestamp(d) <= pd.Timestamp(end_date))]

def _fmt(x):
    return "-" if pd.isna(x) else str(int(x))

def r_between(stock, d0, d1):
    seg = grid.loc[pd.Timestamp(d0):pd.Timestamp(d1), stock].dropna().iloc[1:]
    return (1 + seg / 100).prod() - 1 if len(seg) else float("nan")

for mode in MODES:
    integ = integrate if mode == "count" else integrate_sum
    sched = seoseohi_schedule(base, ksic, all_codes, grid, DEFAULT_BASELINE_DATE,
                              DEFAULT_TRIGGER, end_date, price_gate=False, mode=mode)
    rows = []
    for s, sell_date in sched.items():
        sc = None
        for d in eval_dates:                       # 매도를 일으킨 기준일의 점수 찾기
            cand = _scores(base, ksic, s, DEFAULT_BASELINE_DATE, d)
            if integ(cand) in DEFAULT_TRIGGER:
                sc = cand
                break
        if sc is None:
            continue
        fut = grid.loc[sell_date:, s]
        if end_date is not None:
            fut = fut.loc[:pd.Timestamp(end_date)]
        fut = fut.dropna().iloc[1:]
        if len(fut) == 0:
            continue
        R_hold = (1 + fut / 100).prod() - 1
        R_cash = (1 + CASH_DAILY_RATE) ** len(fut) - 1
        d_, f_, o_ = sc.get("dupont"), sc.get("fscore"), sc.get("ohlson")
        tot = sum(int(v) for v in (d_, f_, o_) if not pd.isna(v))
        sig = "+".join(f"{m}{_fmt(v)}" for m, v in (("D", d_), ("F", f_), ("O", o_))
                       if not pd.isna(v) and int(v) > 0)
        rows.append({"매도일": pd.Timestamp(sell_date).date(), "stock": s,
                     "종목명": name_map.get(s, ""), "매수~매도": r_between(s, entry, sell_date),
                     "판정": GOOD if (R_cash - R_hold) > 0 else BAD, "매도후": R_hold,
                     "D": _fmt(d_), "F": _fmt(f_), "O": _fmt(o_), "합계": tot, "신호": sig})
    if not rows:
        print(f"[{mode}] 매도 없음"); continue
    tbl = pd.DataFrame(rows).sort_values("매도일")
    g, b = (tbl["판정"] == GOOD).sum(), (tbl["판정"] == BAD).sum()
    print(f"\n===== {mode} 방식 : 매도 {len(tbl)}건, 좋은 {g} / 성급 {b}, 적중률 {g / max(g + b, 1):.1%} =====")
    show = tbl.copy()
    show["매수~매도"] = show["매수~매도"].map(lambda x: f"{x:+.1%}")
    show["매도후"] = show["매도후"].map(lambda x: f"{x:+.1%}")
    cols = ["매도일", "stock", "종목명", "매수~매도", "판정", "매도후", "D", "F", "O"]
    cols += (["합계"] if mode == "sum" else []) + ["신호"]
    print(show[cols].to_string(index=False))
    print(f"\n[{mode}] 신호(점수) 조합별 성적:")
    agg = tbl.groupby("신호").agg(건수=("stock", "size"),
                                적중률=("판정", lambda x: (x == GOOD).mean()),
                                매도후중앙=("매도후", "median"))
    agg["적중률"] = agg["적중률"].map(lambda x: f"{x:.0%}")
    agg["매도후중앙"] = agg["매도후중앙"].map(lambda x: f"{x:+.1%}")
    print(agg.sort_values("건수", ascending=False).to_string())


In [ ]:
# 매도 품질 집계: 원시 vs winsorize(이상치 중화) vs 중앙값 — 정직한 보고
import pandas as pd
from backtest.montecarlo import sell_study

ss = sell_study(grid, base, ksic, years=4, seed=0, mode="sum")   # 합산 4점 기준
u = ss.drop_duplicates("stock")                                  # 고유 매도 종목
rh, B = u["R_hold"], u["B"]
B_wins = B + (rh - rh.clip(upper=1.0))                           # R_hold 을 +100%로 캡했을 때의 B

print(f"고유 매도 {len(u)}건  |  적중률 {u['good'].mean():.0%}")
print(f"평균B   원시 {B.mean():+.3f}   winsorize {B_wins.mean():+.3f}   |   중앙B {B.median():+.3f}")
print(f"매도후 수익률   평균 {rh.mean():+.1%}   중앙 {rh.median():+.1%}")
print("\n※ 평균은 078130 같은 희석 이상치 한 건에 휘둘린다. 중앙값·winsorize가 대표적인 매도의 실제 가치.")


In [ ]:
# F-Score 1점 드롭 실험 (파일 영구변경 없이, 이전 패치도 자동 복구)
import backtest.montecarlo as mc
import pandas as pd
from models.seoseohi.pipeline import integrate_sum as _orig_sum   # 진짜 원본
mc.integrate_sum = _orig_sum          # 이전 실험이 남긴 패치가 있으면 여기서 복구

def _sum_nof1(scores):
    s = dict(scores)
    fs = s.get("fscore")
    if pd.notna(fs) and int(fs) == 1:     # F-Score 1점만 합계에서 제외 (NA/None 은 그대로)
        s["fscore"] = 0
    return _orig_sum(s)

def _show(tag):
    print(f"[{tag}] 서서히_합산")
    for label, e, b in [("2016 진입·2년", "2016-01-04", "2016-04-01"),
                        ("2018 진입·2년", "2018-01-02", "2018-04-01")]:
        _, s = mc.run_montecarlo(grid, base, years=2, seed=0,
                                 entry_date=e, baseline_date=b, with_sum=True)
        r = s.loc["서서히_합산"]
        print(f"  {label}: 수익 {r['누적수익률']:+.3f}  MDD {r['MDD']:.3f}  "
              f"Sharpe {r['Sharpe']:.3f}  승률 {r['win_vs_무손절']:.2f}")
    ss = mc.sell_study(grid, base, ksic, years=4, seed=0, mode="sum").drop_duplicates("stock")
    print(f"  4년 고유매도 {len(ss)}건  적중률 {ss['good'].mean():.0%}  "
          f"중앙B {ss['B'].median():+.3f}  최악_매도후 {ss['R_hold'].max():+.1%}")

_show("기존 (F1 포함)")
mc.integrate_sum = _sum_nof1
print()
_show("F1 드롭")
mc.integrate_sum = _orig_sum
print("\n원복 완료")

In [5]:
import os
# 아웃오브샘플 검증: 2020-2026 데이터 (다른 종목 + 다른 기간)
from backtest.engine import load_daily_price
from backtest.montecarlo import run_montecarlo, clear_verdict_cache

BASE26 = os.environ["RAWDATA_DIR"]   # 2020-2026 본데이터

grid26 = load_daily_price(BASE26)
clear_verdict_cache()
print("2020-2026 grid:", grid26.shape, "|",
      grid26.index.min().date(), "~", grid26.index.max().date())

for yrs in [2, 4]:
    _, s = run_montecarlo(grid26, BASE26, years=yrs, seed=0,
                          entry_date="2020-01-02", baseline_date="2020-04-01", with_sum=True)
    print(f"\n=== 2020 진입 · {yrs}년 (아웃오브샘플) ===")
    print(s.round(3).to_string())

print("\n\n===== 워크포워드 (진입 연도별 2년) · 서서히_합산 vs 무손절 =====")
for yr in [2020, 2021, 2022, 2023, 2024]:
    _, s = run_montecarlo(grid26, BASE26, years=2, seed=0,
                          entry_date=f"{yr}-01-02", baseline_date=f"{yr}-04-01", with_sum=True)
    r, m = s.loc["서서히_합산"], s.loc["무손절"]
    print(f"{yr} 진입·2년: 서서히_합산 {r['누적수익률']:+.3f} (MDD {r['MDD']:.3f}) "
          f"vs 무손절 {m['누적수익률']:+.3f} (MDD {m['MDD']:.3f})  승률 {r['win_vs_무손절']:.2f}")

2020-2026 grid: (1684, 120) | 2019-12-30 ~ 2026-11-11

=== 2020 진입 · 2년 (아웃오브샘플) ===
          누적수익률  수익률_중앙값    MDD  Sortino  Sharpe  평균_현금대체  평균_매도수  win_vs_무손절
strategy                                                                     
무손절       0.549    0.590  0.452    1.115   0.792      NaN     0.0         0.0
단순손절     -0.062   -0.080  0.287   -0.221  -0.261   -0.923    17.6         0.0
서서히       0.545    0.588  0.452    1.122   0.798   -0.232     3.6         0.6
서서히_합산    0.515    0.571  0.452    1.092   0.785   -0.333     7.5         0.5

=== 2020 진입 · 4년 (아웃오브샘플) ===
          누적수익률  수익률_중앙값    MDD  Sortino  Sharpe  평균_현금대체  평균_매도수  win_vs_무손절
strategy                                                                     
무손절       0.130    0.127  0.492    0.329   0.240      NaN     0.0         0.0
단순손절     -0.200   -0.184  0.331   -0.433  -0.351   -0.449    18.6         0.1
서서히       0.170    0.161  0.467    0.363   0.266   -0.073     6.6         0.9
서서히_합산    0.222    0.204  

# Stress Testing

In [6]:
import os
# Stress testing on the OUT-OF-SAMPLE data (2020-2026). Model untouched — measurement only.
BASE26 = os.environ["RAWDATA_DIR"]   # 2020-2026 본데이터
from backtest.engine import load_daily_price
from backtest.montecarlo import load_ksic_from_base
grid26  = load_daily_price(BASE26)
ksic26  = load_ksic_from_base(BASE26)
print("grid26:", grid26.shape, grid26.index.min().date(), "~", grid26.index.max().date())

grid26: (1684, 120) 2019-12-30 ~ 2026-11-11


In [7]:
import numpy as np, pandas as pd
from backtest.engine import run_strategy
from backtest.montecarlo import (seoseohi_schedule, stoploss_schedule,
                                  DEFAULT_TRIGGER, DEFAULT_STOP_PCT)
from models.seoseohi.fscore import list_snapshot_dates
from backtest.metrics import cumret, mdd, sortino, sharpe

def _baseline_after(base_dir, entry):
    """서서히 기준선 = 진입일 직후 첫 스냅샷(그 때부터 재무 훼손을 지켜본다)."""
    snaps = sorted(pd.Timestamp(d) for d in list_snapshot_dates(base_dir))
    after = [d for d in snaps if d > pd.Timestamp(entry)]
    return after[0] if after else None

def shift_schedule(sched, days, lag):
    """매도일을 lag 거래일만큼 뒤로 민다 (T+1 체결 모사)."""
    if lag == 0: return dict(sched)
    out = {}
    for s, d in sched.items():
        pos = min(days.searchsorted(pd.Timestamp(d)) + lag, len(days) - 1)
        out[s] = days[pos]
    return out

def random_sell_schedule(codes, k, window, rng):
    """합산이 판 '개수'와 똑같이, 아무 종목이나 아무 날에 판다 (타이밍 실력 비교용)."""
    if k <= 0 or len(window) == 0: return {}
    picks = rng.choice(list(codes), size=min(k, len(codes)), replace=False)
    return {s: window[rng.integers(0, len(window))] for s in picks}

def run_stress(grid, base_dir, ksic, n_runs=120, n_pick=20, horizon_years=2,
               seed=7, trigger=DEFAULT_TRIGGER, price_gate=False,
               lags=(0, 1), n_rand=3, entry_min="2020-01-02"):
    """랜덤 진입일 × 랜덤 바스켓. 한 런마다 스케줄을 '한 번만' 계산하고,
    그 위에서 (각 lag 전략) + (무작위매도)를 평가한다. 점수계산이 병목이라 공유한다."""
    rng  = np.random.default_rng(seed)
    days = grid.index
    H    = pd.DateOffset(years=horizon_years)
    elig = days[(days >= pd.Timestamp(entry_min)) & (days <= days[-1] - H)]
    codes_all = list(grid.columns)
    rows = []
    for run_id in range(n_runs):
        entry = elig[rng.integers(0, len(elig))]
        end   = entry + H
        base  = _baseline_after(base_dir, entry)
        if base is None or base >= end:
            continue
        codes  = list(rng.choice(codes_all, size=n_pick, replace=False))
        basket = grid[codes]
        win    = days[(days >= base) & (days <= end)]
        # 비싼 부분: 스케줄 1회 계산 (count / sum)
        sched = {
            "무손절": {},
            "단순손절": stoploss_schedule(basket, DEFAULT_STOP_PCT, start=entry, end_date=end),
            "서서히":   seoseohi_schedule(base_dir, ksic, codes, grid, baseline_date=base,
                                        trigger=trigger, end_date=end, price_gate=price_gate, mode="count"),
            "서서히_합산": seoseohi_schedule(base_dir, ksic, codes, grid, baseline_date=base,
                                        trigger=trigger, end_date=end, price_gate=price_gate, mode="sum"),
        }
        # 싼 부분: 각 lag × 전략 재무+가격 곡선
        for lag in lags:
            for name, sc in sched.items():
                sc_l = {} if name == "무손절" else shift_schedule(sc, days, lag)
                W = run_strategy(basket, sc_l, end_date=end, start_date=entry)
                rows.append({"run": run_id, "entry": entry, "strategy": name, "lag": lag,
                             "누적수익률": cumret(W), "MDD": mdd(W),
                             "Sortino": sortino(W), "Sharpe": sharpe(W), "매도수": len(sc_l)})
        # 무작위매도 (합산 매도수에 맞춰, n_rand회 평균)
        k  = len(sched["서서히_합산"])
        rr = [cumret(run_strategy(basket, random_sell_schedule(codes, k, win, rng),
                                  end_date=end, start_date=entry)) for _ in range(n_rand)]
        rows.append({"run": run_id, "entry": entry, "strategy": "무작위매도", "lag": 0,
                     "누적수익률": float(np.mean(rr)) if rr else np.nan,
                     "MDD": np.nan, "Sortino": np.nan, "Sharpe": np.nan, "매도수": k})
    return pd.DataFrame(rows)

# 네 가지 해석 (detail 테이블에서 즉시 뽑는다)
def _piv(detail, lag=0, val="누적수익률"):
    return detail[detail.lag == lag].pivot_table(index="run", columns="strategy", values=val)

def test1_distribution(detail):
    p = _piv(detail); m = _piv(detail, val="MDD")
    print(f"=== Test 1 · 랜덤 진입일+랜덤 바스켓 분포  (n={p.shape[0]} 런) ===")
    for s in ["무손절", "서서히", "서서히_합산"]:
        if s not in p: continue
        win = (p[s] > p["무손절"]).mean()
        print(f"  {s:9s} 중앙값 {p[s].median():+.3f}  [10%p {p[s].quantile(.1):+.3f}, "
              f"90%p {p[s].quantile(.9):+.3f}]  MDD중앙값 {m[s].median():.3f}  무손절대비승률 {win:.2f}")

def test2_random_sell(detail):
    p = _piv(detail)[["서서히_합산", "무작위매도"]].dropna()
    edge = p["서서히_합산"] - p["무작위매도"]
    print("=== Test 2 · 타이밍에 실력이 있나 — 서서히_합산 vs 무작위매도(같은 매도수) ===")
    print(f"  합산이 무작위매도를 이긴 비율: {(edge>0).mean():.2f}  (n={len(p)})")
    print(f"  평균 우위 {edge.mean():+.3f} · 중앙값 {edge.median():+.3f}")

def test3_left_tail(detail, q=5):
    p = _piv(detail).dropna(subset=["무손절", "서서히_합산"]); m = _piv(detail, val="MDD")
    p = p.assign(bucket=pd.qcut(p["무손절"], q, duplicates="drop"))
    print("=== Test 3 · 보험 성격 — 무손절이 나쁜 구간일수록 합산이 더 돕나 ===")
    print(f"  {'무손절버킷(수익률)':24s}{'무손절평균':>10s}{'합산엣지':>10s}{'MDD감소':>10s}")
    for b, g in p.groupby("bucket", observed=True):
        edge = (g["서서히_합산"] - g["무손절"]).mean()
        mred = (m.loc[g.index, "무손절"] - m.loc[g.index, "서서히_합산"]).mean()
        print(f"  {str(b):24s}{g['무손절'].mean():>+10.3f}{edge:>+10.3f}{mred:>+10.3f}")

def test4_lag(detail):
    print("=== Test 4 · T+1 체결 지연이 엣지를 갉아먹나 ===")
    for s in ["서서히", "서서히_합산"]:
        parts = []
        for lag in sorted(detail.lag.unique()):
            p = _piv(detail, lag=lag)
            if s in p and "무손절" in p:
                parts.append(f"lag{lag}: 엣지 {(p[s]-p['무손절']).mean():+.3f} / 승률 {(p[s]>p['무손절']).mean():.2f}")
        print(f"  {s:9s} " + "   ".join(parts))

In [10]:
detail = run_stress(grid26, BASE26, ksic26, n_runs=200, horizon_years=2, lags=(0, 1), seed=7)
detail.to_csv("stress_detail.csv", index=False)
print("done:", detail.shape, "| 전략들:", sorted(detail.strategy.unique()))

done: (1800, 9) | 전략들: ['단순손절', '무손절', '무작위매도', '서서히', '서서히_합산']


In [11]:
test1_distribution(detail)
print()
test2_random_sell(detail)
print()
test3_left_tail(detail)
print()
test4_lag(detail)

=== Test 1 · 랜덤 진입일+랜덤 바스켓 분포  (n=200 런) ===
  무손절       중앙값 +0.049  [10%p -0.251, 90%p +0.808]  MDD중앙값 0.352  무손절대비승률 0.00
  서서히       중앙값 +0.029  [10%p -0.224, 90%p +0.748]  MDD중앙값 0.306  무손절대비승률 0.67
  서서히_합산    중앙값 +0.070  [10%p -0.203, 90%p +0.815]  MDD중앙값 0.264  무손절대비승률 0.69

=== Test 2 · 타이밍에 실력이 있나 — 서서히_합산 vs 무작위매도(같은 매도수) ===
  합산이 무작위매도를 이긴 비율: 0.62  (n=200)
  평균 우위 +0.028 · 중앙값 +0.037

=== Test 3 · 보험 성격 — 무손절이 나쁜 구간일수록 합산이 더 돕나 ===
  무손절버킷(수익률)                   무손절평균      합산엣지     MDD감소
  (-0.412, -0.178]            -0.264    +0.124    +0.088
  (-0.178, -0.0443]           -0.117    +0.087    +0.092
  (-0.0443, 0.147]            +0.047    +0.024    +0.096
  (0.147, 0.478]              +0.283    +0.009    +0.087
  (0.478, 1.619]              +0.890    -0.046    +0.047

=== Test 4 · T+1 체결 지연이 엣지를 갉아먹나 ===
  서서히       lag0: 엣지 +0.001 / 승률 0.67   lag1: 엣지 +0.004 / 승률 0.68
  서서히_합산    lag0: 엣지 +0.039 / 승률 0.69   lag1: 엣지 +0.042 / 승률 0.70


# Stress Test Across Multiple Number of Years

In [12]:
def run_stress_multi(grid, base_dir, ksic, horizons=(1, 2, 3, 4),
                     n_runs=150, seed=7, **kw):
    """같은 설정으로 보유연수만 바꿔가며 돌리고 'horizon' 태그를 붙여 합친다.
    점수 캐시(_VERDICT_CACHE)가 보유연수 간에 공유되므로 뒤쪽 horizon 은 빨라진다."""
    parts = []
    for h in horizons:
        d = run_stress(grid, base_dir, ksic, n_runs=n_runs, horizon_years=h, seed=seed, **kw)
        d["horizon"] = h
        parts.append(d)
        print(f"  horizon {h}년 done ({d.run.nunique()} 런)")
    return pd.concat(parts, ignore_index=True)

def summary_by_horizon(detail):
    print(f"{'보유연수':>6s}{'무손절중앙':>11s}{'합산중앙':>10s}{'합산MDD':>9s}"
          f"{'무손절MDD':>10s}{'MDD감소':>9s}{'무손절대비승률':>13s}{'vs무작위승률':>12s}")
    for h, d in detail.groupby("horizon"):
        d0 = d[d.lag == 0]
        p = d0.pivot_table(index="run", columns="strategy", values="누적수익률")
        m = d0.pivot_table(index="run", columns="strategy", values="MDD")
        win_bh   = (p["서서히_합산"] > p["무손절"]).mean()
        vr       = p[["서서히_합산", "무작위매도"]].dropna()
        win_rand = (vr["서서히_합산"] > vr["무작위매도"]).mean()
        mred     = (m["무손절"] - m["서서히_합산"]).median()
        print(f"{h:>6d}{p['무손절'].median():>+11.3f}{p['서서히_합산'].median():>+10.3f}"
              f"{m['서서히_합산'].median():>9.3f}{m['무손절'].median():>10.3f}"
              f"{mred:>+9.3f}{win_bh:>13.2f}{win_rand:>12.2f}")

In [13]:
multi = run_stress_multi(grid26, BASE26, ksic26, horizons=(1, 2, 3, 4), n_runs=150, seed=7)
multi.to_csv("stress_multi.csv", index=False)
summary_by_horizon(multi)

  horizon 1년 done (150 런)
  horizon 2년 done (150 런)
  horizon 3년 done (150 런)
  horizon 4년 done (150 런)
  보유연수      무손절중앙      합산중앙    합산MDD    무손절MDD    MDD감소      무손절대비승률     vs무작위승률
     1     +0.077    +0.081    0.202     0.236   +0.013         0.57        0.58
     2     +0.033    +0.054    0.268     0.344   +0.060         0.68        0.62
     3     -0.003    +0.040    0.304     0.418   +0.097         0.69        0.56
     4     -0.047    +0.018    0.328     0.479   +0.130         0.69        0.55


# Across All Years 2

In [14]:
def run_stress_multi(grid, base_dir, ksic, horizons=(1, 2, 3, 4, 5, 6),
                     n_runs=150, seed=7, **kw):
    """보유연수만 바꿔가며 돌려 'horizon' 태그로 합친다. 점수 캐시가 연수 간 공유돼 뒤쪽이 빠르다.
    (주의) 긴 보유연수일수록 '진입+연수 ≤ 데이터끝' 때문에 진입일이 2020년에 몰린다.
            6년 = 사실상 '2020년 진입·전구간 보유'."""
    parts = []
    for h in horizons:
        d = run_stress(grid, base_dir, ksic, n_runs=n_runs, horizon_years=h, seed=seed, **kw)
        d["horizon"] = h
        parts.append(d)
        print(f"  horizon {h}년 done ({d.run.nunique()} 런)")
    return pd.concat(parts, ignore_index=True)

def summary_by_horizon(detail):
    """보유연수별로 서서히_합산 vs 무손절(기준) 핵심 지표를 한 표로. 모두 런별 중앙값/승률."""
    base, S = "무손절", "서서히_합산"
    recs = []
    for h, d in detail.groupby("horizon"):
        d0 = d[d.lag == 0]
        R  = d0.pivot_table(index="run", columns="strategy", values="누적수익률")
        M  = d0.pivot_table(index="run", columns="strategy", values="MDD")
        SH = d0.pivot_table(index="run", columns="strategy", values="Sharpe")
        SO = d0.pivot_table(index="run", columns="strategy", values="Sortino")
        vr = R[[S, "무작위매도"]].dropna()
        recs.append({
            "보유연수": f"{h}년", "런수": int(R.shape[0]),
            "수익_무손절": R[base].median(),      "수익_합산": R[S].median(),
            "수익개선(합산-무손절)": (R[S] - R[base]).median(),
            "MDD_무손절": M[base].median(),       "MDD_합산": M[S].median(),
            "MDD개선(낮을수록好)": (M[base] - M[S]).median(),
            "Sharpe_무손절": SH[base].median(),   "Sharpe_합산": SH[S].median(),
            "Sharpe개선": (SH[S] - SH[base]).median(),
            "Sortino_무손절": SO[base].median(),  "Sortino_합산": SO[S].median(),
            "Sortino개선": (SO[S] - SO[base]).median(),
            "승률_vs무손절": (R[S] > R[base]).mean(),
            "승률_vs무작위매도": (vr[S] > vr["무작위매도"]).mean(),
        })
    tbl = pd.DataFrame(recs).set_index("보유연수").round(3)
    return tbl.T          # 측정치=행, 보유연수=열 → 연수에 따른 추세를 가로로 읽는다

In [15]:
pd.set_option("display.max_columns", None, "display.width", 200)
multi = run_stress_multi(grid26, BASE26, ksic26, horizons=(1, 2, 3, 4, 5, 6), n_runs=150, seed=7)
multi.to_csv("stress_multi.csv", index=False)
summary_by_horizon(multi)      # 셀 마지막 줄 → 표로 렌더링

  horizon 1년 done (150 런)
  horizon 2년 done (150 런)
  horizon 3년 done (150 런)
  horizon 4년 done (150 런)
  horizon 5년 done (150 런)
  horizon 6년 done (150 런)


보유연수,1년,2년,3년,4년,5년,6년
런수,150.000,150.000,150.000,150.000,150.000,150.000
수익_무손절,0.077,0.033,-0.003,-0.047,-0.088,0.128
수익_합산,0.081,0.054,0.040,0.018,0.004,0.233
수익개선(합산-무손절),0.009,0.040,0.075,0.117,0.130,0.126
MDD_무손절,0.236,0.344,0.418,0.479,0.541,0.551
MDD_합산,0.202,0.268,0.304,0.328,0.359,0.390
MDD개선(낮을수록好),0.013,0.060,0.097,0.130,0.155,0.161
Sharpe_무손절,0.422,0.246,0.188,0.139,0.147,0.240
Sharpe_합산,0.449,0.264,0.251,0.176,0.138,0.295
Sharpe개선,0.013,0.058,0.049,0.049,0.031,0.039
